In [ ]:
import xarray as xr
import numpy as np
from scipy.ndimage import minimum_filter
from scipy.ndimage import label, generate_binary_structure
import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
def crosses_dateline(labeled_arr, resolution):
    #Shift 180 degrees
    shifted_arr = np.roll(labeled_arr, 180*resolution, axis = 1)

    #Examine a 3x3 region centered at the dateline
    for lat in range(0, len(shifted_arr) - 1):
    	sample_arr = shifted_arr[lat:lat+2, 179:181]

    	top_left = sample_arr[0, 0]
    	top_right = sample_arr[0, 1]
    	bottom_right = sample_arr[1, 1]
    	bottom_left = sample_arr[1, 0]
		
    	#Compare top left with position next to it
    	if(top_left != 0 and top_right != 0):
    		if(top_left != top_right):
    			num_to_change = top_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, top_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]

    	#Compare top left with position diagonal to it
    	if(top_left != 0 and bottom_right != 0):
    		if(top_left != bottom_right):
    			num_to_change = top_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, bottom_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]
		
    	#Compate bottom left with position next to it
    	if(bottom_left != 0 and bottom_right != 0):
    		if(bottom_left != bottom_right):
    			num_to_change = bottom_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, bottom_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]
    			#print('Changed arr bot left, bot right', sample_arr)
    	
    	#Compare bottom left with position diagonal to it
    	if(bottom_left != 0 and top_right != 0):
    		if(bottom_left != top_right):
    			num_to_change = bottom_left
    			shifted_arr = np.where(shifted_arr != num_to_change, shifted_arr, top_right)
    			sample_arr = shifted_arr[lat:lat+2, 179:181]
    		
    return np.roll(shifted_arr, 180, axis = 1)
    
    

In [ ]:
ds = xr.open_dataset('../notebooks/data/SLP_ex.nc')

In [ ]:
ds

#Just plot a sample time step of MSLP. Could also plot a loop
plt.pcolormesh(ds["longitude"].values, ds["latitude"].values, ds['SLP'].values[0])
plt.colorbar()

In [ ]:
SLP = ds['SLP']

SLP_hPa = SLP / 100

In [ ]:
SLP_arr = SLP_hPa.to_numpy()

In [ ]:
print(SLP_arr)

In [ ]:
SLP_thresh = 1000 #Units of hPa

SLP_1000 = np.where(SLP_arr < SLP_thresh, SLP, 0)

In [ ]:
print(SLP_1000)

In [ ]:
s = generate_binary_structure(2,2)

In [ ]:
resolution = 1 #In degrees

labeled_arr = np.zeros_like(SLP_1000)

num_features_arr = np.zeros(len(SLP_1000))

for time_step in np.arange(0, len(SLP_1000)):
    labeled_SLP_field, num_features = label(SLP_1000[time_step], s)
  
    correct_labeled_SLP_field = crosses_dateline(labeled_SLP_field, resolution)
    
    num_features_arr[time_step] = num_features

    labeled_arr[time_step] = correct_labeled_SLP_field

    

In [ ]:
#Plot the labeled pfield. Could also plot a loop

plt.pcolormesh(ds["longitude"].values, ds["latitude"].values, labeled_arr[0])
plt.colorbar()